# BTC Oracle: LSTM Deep Learning Model

This notebook implements a Deep Learning approach (LSTM) to predict Bitcoin price movements using the **BTC Oracle** dataset.

## Goal
Achieve high-precision predictions by training on a rich dataset containing:
- **Price Data:** OHLCV
- **On-Chain Metrics:** Hash Rate, Difficulty, etc.
- **Sentiment:** Fear & Greed Index
- **Macro:** S&P 500, DXY, VIX, etc.

We aim for **High Confidence** accuracy > 80-90% by allowing the model to only trade when it is certain.

In [ ]:
# Install kagglehub if not present
!pip install kagglehub -q

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix

# Configure plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading via KaggleHub
Automatically download the latest version of the dataset.

In [ ]:
# Download latest version
path = kagglehub.dataset_download("oussamataghlaoui/btc-oracle-on-chain-sentiment-and-macro-data")

print("Path to dataset files:", path)

# Find the CSV file automatically
import os
import glob

csv_files = glob.glob(os.path.join(path, "*.csv"))
if csv_files:
    dataset_path = csv_files[0]
    print(f"Found dataset: {dataset_path}")
    # FIX: Column is named 'Datetime' not 'timestamp'
    df = pd.read_csv(dataset_path, parse_dates=['Datetime'], index_col='Datetime')
else:
    raise FileNotFoundError("No CSV file found in the downloaded path!")

print(f"Data Shape: {df.shape}")
df.head()

## 2. Feature Engineering & Target Creation

We use a **Percentile-based Target**. Instead of predicting simple Up/Down (which is noisy), we predict:
- **Class 1 (UP):** Return > Xth percentile
- **Class 0 (DOWN):** Return < Yth percentile
- **Class 2 (NEUTRAL):** Small moves (Ignored or Separate Class)

For this model, we will use a **3-class system** but focus on correctly identifying the Up/Down moves.

In [ ]:
def create_percentile_target(df, return_col='future_return_24h', percentile=65):
    """
    Excludes neutral moves to focus on significant trends.
    """
    data = df.copy()
    returns = data[return_col]
    
    up_thresh = np.percentile(returns.dropna(), percentile)
    down_thresh = np.percentile(returns.dropna(), 100 - percentile)
    
    print(f"Thresholds | UP > {up_thresh:.4f} | DOWN < {down_thresh:.4f}")
    
    # Initialize as Neutral (Class 1)
    data['target'] = 1 
    
    # Class 2: Big UP
    data.loc[returns > up_thresh, 'target'] = 2
    
    # Class 0: Big DOWN
    data.loc[returns < down_thresh, 'target'] = 0
    
    return data.dropna(subset=[return_col]), (down_thresh, up_thresh)

# Create target based on 24h future returns
df_clean, thresholds = create_percentile_target(df, 'future_return_24h', percentile=60)

print("Class Distribution:")
print(df_clean['target'].value_counts(normalize=True))

## 3. Preprocessing: Scaling & Sequence Generation
LSTMs need sequences (e.g., past 24 hours of data) to predict the next step.

In [ ]:
def create_sequences(features, targets, seq_length=24):
    X, y = [], []
    for i in range(len(features) - seq_length):
        X.append(features[i:i+seq_length])
        y.append(targets[i+seq_length])
    return np.array(X), np.array(y)

# Select Features (Exclude targets and future cols)
feature_cols = [c for c in df_clean.columns if 'future' not in c and 'target' not in c and 'return' not in c]
# FIX: Keep only numeric columns (exclude categorical like 'fear_greed_classification')
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_clean[c])]
print(f"Using {len(feature_cols)} numeric features.")

# Split Data (Time-series split)
train_size = int(len(df_clean) * 0.7)
val_size = int(len(df_clean) * 0.15)

train_df = df_clean.iloc[:train_size]
val_df = df_clean.iloc[train_size:train_size+val_size]
test_df = df_clean.iloc[train_size+val_size:]

# Scaling (Fit on Train ONLY)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(train_df[feature_cols])
X_val_scaled = scaler.transform(val_df[feature_cols])
X_test_scaled = scaler.transform(test_df[feature_cols])

# Create Sequences
SEQ_LEN = 48 # Lookback 48 hours

X_train, y_train = create_sequences(X_train_scaled, train_df['target'].values, SEQ_LEN)
X_val, y_val = create_sequences(X_val_scaled, val_df['target'].values, SEQ_LEN)
X_test, y_test = create_sequences(X_test_scaled, test_df['target'].values, SEQ_LEN)

print(f"Train Shape: {X_train.shape}")
print(f"Val Shape: {X_val.shape}")
print(f"Test Shape: {X_test.shape}")

## 4. LSTM Model Architecture
We use a Bidirectional LSTM with Dropout to prevent overfitting.

In [ ]:
input_shape = (X_train.shape[1], X_train.shape[2])

model = tf.keras.Sequential([
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True), input_shape=input_shape),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax') # 3 Classes: Down(0), Neutral(1), Up(2)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

## 5. Evaluation & High Confidence Analysis
We evaluate standard accuracy, but more importantly, we check the accuracy when the model is **>90% confident**.

In [ ]:
# Plot History
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss Curve")

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title("Accuracy Curve")
plt.show()

In [ ]:
# Predictions
y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)
y_conf = np.max(y_prob, axis=1)

print("--- Standard Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['DOWN', 'NEUTRAL', 'UP']))

# High Confidence Analysis
CONFIDENCE_THRESHOLD = 0.85

high_conf_mask = y_conf >= CONFIDENCE_THRESHOLD
y_test_hc = y_test[high_conf_mask]
y_pred_hc = y_pred[high_conf_mask]

print(f"\n--- High Confidence Report (> {CONFIDENCE_THRESHOLD*100:.0f}%) ---")
print(f"Coverage: {np.sum(high_conf_mask)} / {len(y_test)} samples ({np.sum(high_conf_mask)/len(y_test):.1%})")

if len(y_test_hc) > 0:
    print(classification_report(y_test_hc, y_pred_hc, target_names=['DOWN', 'NEUTRAL', 'UP']))
    
    # Confusion Matrix for HC
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y_test_hc, y_pred_hc), annot=True, fmt='d', cmap='Blues')
    plt.title("High Confidence Confusion Matrix")
    plt.show()
else:
    print("No predictions met the confidence threshold.")